## Задание 1. Обучите нейронную сеть решать шифр Цезаря.

### 1. `Функция шифра Цезаря`

In [1]:
# Функция которая шифрует текс(строки). K = порядок смещения
def caesar_cipher(text, k):
    alphabet = 'АБВГДЕЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯ'
    result = ''
    for char in text:
        if char in alphabet:
            index = alphabet.find(char)
            new_index = (index + k) % len(alphabet) # длина алфавита. % оператор по модулю, возвращает целый остаток
            result += alphabet[new_index]
        else:
            result += char
    return result


### 2. `Генерация обучающих и тестовых данных`

In [2]:
# Функция для генерации данных как для обучения так и для проверки (теста)
import random
def generate_data(n=1000):
    X, y = [], [] # списки зашифрованных и расшифрованных слов соотвественно
    alphabet = 'АБВГДЕЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯ'

    for _ in range(n): # количество слов
        # случайное слово из 5–12 букв
        length = random.randint(5, 12) # определяем случайную длину слова
        word = ''.join(random.choice(alphabet) for _ in range(length)) # определяет рандомно по одному из букв алфавита, длина слова определена переменной length

        # По умолчании k = 2.
        shift = 2 #random.randint(1, 3) #Случайный сдвиг от 1 до 32 (к = определяем случайным образом)

        # Шифруем слово функцией caesar_cipher
        encrypted_word = caesar_cipher(word, shift)

        X.append(encrypted_word)  # Список зашифрованных слов (вход для модели)
        y.append(word)            # Список расшифрованных  (выход модели)

    return X, y

In [3]:
# Получаем данные для обучения и теста
X_train, y_train = generate_data(1200)  # 800 для обучения
X_test, y_test = generate_data(240)     # 200 для проверки

In [4]:
X_train[:5]

['ЭНЕРЪ', 'ДЩЖШЪЫЧЯВТБШ', 'ЙЭОКЧД', 'ОДГЗЩСЦЙ', 'КШАМЭЙЮШТ']

### 3. `Подготовка данных`/ оцифровка букв

In [5]:
alphabet = 'АБВГДЕЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯ'
char_to_idx = {char: i for i, char in enumerate(alphabet)} # словарь, где ключ - буква, а значение индекс соотвествующей буквы словаря
idx_to_char = {i: char for i, char in enumerate(alphabet)} # словарь, где ключ - индекс, а значение соотвествующая буква словаря

def text_to_numbers(text):
    return [char_to_idx[char] for char in text]

# Преобразуем все тексты в числовые последовательности
X_train_num = [text_to_numbers(text) for text in X_train]
y_train_num = [text_to_numbers(text) for text in y_train]
X_test_num = [text_to_numbers(text) for text in X_test]
y_test_num = [text_to_numbers(text) for text in y_test]

### 4. `RNN` Нейронная сеть.



In [6]:
import torch.nn as nn

class decoderCaesarNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(33, 64)  # 33 буквы, 64 — размерность вектора
        self.lstm = nn.RNN(64, 128, batch_first=True)  # LSTM слой
        self.fc = nn.Linear(128, 33)  # выходной слой (33 буквы)

    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.lstm(x)
        x = self.fc(x)
        return x

model = decoderCaesarNet()

### 5. `Обучение модели`

In [7]:
import torch
# Функция потерь и оптимизатор
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Обучение на 50 эпохах
for epoch in range(50):
    total_loss = 0
    for i in range(len(X_train_num)):
        # Берём одно слово
        input_seq = torch.tensor([X_train_num[i]], dtype=torch.long)
        target_seq = torch.tensor([y_train_num[i]], dtype=torch.long)

        # Обнуляем градиенты
        optimizer.zero_grad()

        # Пропускаем через сеть
        output = model(input_seq)

        # Считаем ошибку
        loss = criterion(output.view(-1, 33), target_seq.view(-1))

        # Обратное распространение ошибки
        loss.backward()

        # Обновляем веса
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f'Эпоха {epoch + 1}, потеря: {total_loss / len(X_train_num):.4f}')


Эпоха 10, потеря: 0.0000
Эпоха 20, потеря: 0.0000
Эпоха 30, потеря: 0.0000
Эпоха 40, потеря: 0.0000
Эпоха 50, потеря: 0.0000


### 6. `Проверка качества модели`

In [8]:
X_test[5]

'ДЭУМЖ'

In [9]:
def numbers_to_text(numbers):
    return ''.join([idx_to_char[num] for num in numbers])

# Тестируем на нескольких примерах
model.eval()  # режим оценки
print("\nПримеры работы:")

# Переменные для расчёта test_loss и test_accuracy
test_loss = 0.0
correct_predictions = 0
total_characters = 0

# Функция потерь для расчёта test_loss (должна совпадать с функцией потерь при обучении)
criterion_test = nn.CrossEntropyLoss()

for i in range(len(X_test)):  # Проход по всему тестовому набору (X_test_num, y_test_num - данные уже были ранее преобразованы. Это пример 2 варианта)

    encrypted_word = X_test[i] # зашифрованное слово
    true_word = y_test[i]      # правильный ответ

    # Преобразование в числа
    input_seq = torch.tensor([text_to_numbers(encrypted_word)], dtype=torch.long)
    target_seq = torch.tensor([text_to_numbers(true_word)], dtype=torch.long)  # Целевая последовательность в числовом формате

    # Предсказание модели
    with torch.no_grad():
        output = model(input_seq)
        # Расчёт ошибки для текущего примера
        loss = criterion_test(output.view(-1, 33), target_seq.view(-1))
        test_loss += loss.item()

        # Получаем предсказанные индексы
        predicted_indices = torch.argmax(output, dim=2)[0].numpy()

    # Перевод обратно в текст
    predicted_word = numbers_to_text(predicted_indices)

    # Примеры (только первые 3)
    if i < 3:
        print(f"Зашифровано: {encrypted_word}")
        print(f"Правильно:   {true_word}")
        print(f"Предсказано: {predicted_word}")
        print("-" * 20)

    # Расчёт точности: сравнение предсказанные и истинные символы
    true_indices = text_to_numbers(true_word)
    correct_predictions += sum(1 for pred, true in zip(predicted_indices, true_indices) if pred == true)
    total_characters += len(true_indices)

# Финальный расчёт test_loss и test_accuracy
test_loss /= len(X_test)
test_accuracy = correct_predictions / total_characters if total_characters > 0 else 0

# Вывод итоговых метрик
print(f"\nИтоговая тестовая потеря (test_loss): {test_loss:.4f}")
print(f"Итоговая точность на тестовом наборе (test_accuracy): {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)")



Примеры работы:
Зашифровано: ЩЫГЩНЙШ
Правильно:   ЧЩБЧЛЗЦ
Предсказано: ЧЩБЧЛЗЦ
--------------------
Зашифровано: НПКЗЖХУИМЗАЦ
Правильно:   ЛНИЕДУСЖКЕЮФ
Предсказано: ЛНИЕДУСЖКЕЮФ
--------------------
Зашифровано: ЬЮТДЖБЛД
Правильно:   ЪЬРВДЯЙВ
Предсказано: ЪЬРВДЯЙВ
--------------------

Итоговая тестовая потеря (test_loss): 0.0000
Итоговая точность на тестовом наборе (test_accuracy): 1.0000 (100.00%)


# Задание 2. Выполнить практическую работу из лекционного ноутбука

In [1]:
import pandas as pd, google
import torch

In [2]:
filepath = 'D:/Temp/Учёба/Домашнее_Задание16. Deep Learning/ДЗ_4_Рекуррентные сети/data/data.csv'

In [3]:
uploaded = google.colab.files.upload()

Saving data.csv to data.csv


In [36]:
df = pd.read_csv(filepath_or_buffer='data.csv', sep=',')
df.head(5)

,Unnamed: 0,id,episode_id,number,raw_text,timestamp_in_ms,speaking_line,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
0,0,10368,35,29,"Lisa Simpson: Maggie, look. What's that?",235000,True,9,5.0,Lisa Simpson,Simpson Home,"Maggie, look. What's that?",maggie look whats that,4.0
1,1,10369,35,30,Lisa Simpson: Lee-mur. Lee-mur.,237000,True,9,5.0,Lisa Simpson,Simpson Home,Lee-mur. Lee-mur.,lee-mur lee-mur,2.0
2,2,10370,35,31,Lisa Simpson: Zee-boo. Zee-boo.,239000,True,9,5.0,Lisa Simpson,Simpson Home,Zee-boo. Zee-boo.,zee-boo zee-boo,2.0
3,3,10372,35,33,Lisa Simpson: I'm trying to teach Maggie that ...,245000,True,9,5.0,Lisa Simpson,Simpson Home,I'm trying to teach Maggie that nature doesn't...,im trying to teach maggie that nature doesnt e...,24.0
4,4,10374,35,35,"Lisa Simpson: It's like an ox, only it has a h...",254000,True,9,5.0,Lisa Simpson,Simpson Home,"It's like an ox, only it has a hump and a dewl...",its like an ox only it has a hump and a dewlap...,18.0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11639 entries, 0 to 11638
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Unnamed: 0          11639 non-null  int64  
 1   id                  11639 non-null  int64  
 2   episode_id          11639 non-null  int64  
 3   number              11639 non-null  int64  
 4   raw_text            11639 non-null  object 
 5   timestamp_in_ms     11639 non-null  int64  
 6   speaking_line       11639 non-null  bool   
 7   character_id        11639 non-null  int64  
 8   location_id         11622 non-null  float64
 9   raw_character_text  11639 non-null  object 
 10  raw_location_text   11622 non-null  object 
 11  spoken_words        10893 non-null  object 
 12  normalized_text     10891 non-null  object 
 13  word_count          10893 non-null  float64
dtypes: bool(1), float64(2), int64(6), object(5)
memory usage: 1.2+ MB


## Подготовка данных

In [14]:
# Оставим только достаточно длинные реплики
# df = df.loc[df['normalized_text'].str.len() > 10]

In [5]:
# Соединяем все строки списка в одну строку
all_text = ' '.join(df['normalized_text'].dropna().tolist()) # так как есть пропуски, пропуски удалены

In [6]:
all_text

'maggie look whats that lee-mur lee-mur zee-boo zee-boo im trying to teach maggie that nature doesnt end with the barnyard i want her to have all the advantages that i didnt have its like an ox only it has a hump and a dewlap hump and dew-lap hump and dew-lap you know his blood type how romantic oh yeah whats my shoe size ring yes dad ooh look maggie what is that do-dec-ah-edron dodecahedron thats okay bart nobody really believed it we were just trying to scare you wait dad hes smiling its to open the crate stupid no maggie not az-tec ol-mec ol-mec perhaps there is no moral to this story bart bart hey bart no no hes fine bart my birthday is in two days im gonna be eight years old its a big number -- almost double digits bart will you please let me pour my little heart out bart i do so much for you and yet you have disappointed me on every one of my birthdays ive made things for you but youve lost or broken them in hours but okay well forget all oh thank you well all right -- if you lis

In [7]:
# построение словаря всех символов (character-level vocabulary)
chars = sorted(list(set(all_text)))
print(chars)

[' ', '-', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'È', 'á', 'â', 'ä', 'å', 'ç', 'é', 'ê', 'ë', 'í', 'ï', 'ñ', 'ó', 'ö', 'ú', 'ü']


In [8]:
dictionary_size = len(chars)
print(f"Размер словаря: {dictionary_size} с выражениями героев сериала 'Симпсоны'")

Размер словаря: 55 с выражениями героев сериала 'Симпсоны'


In [9]:
# Cловарь символов
char_to_idx = {char: index for index, char in enumerate(chars)}
idx_to_char = {index: char for index, char in enumerate(chars)}

## Создание DataLoader для модели

In [10]:
# Параметры
SEQ_LENGTH = 50 # длина последовательности
BATCH_SIZE = 64 # размер батча

In [11]:
# Преобразуем весь текст в последовательность индексов использую словарь символов char_to_idx
data_indices = [char_to_idx[ch] for ch in all_text if ch in char_to_idx]
print(f'Общее число индексов всего текста - all_text: {len(data_indices)}')

Общее число индексов всего текста - all_text: 515866


In [12]:
# Списки последовательностей
input_sequences = []
target_sequences = []

In [13]:
for i in range(0, len(data_indices) - SEQ_LENGTH): # 0 и до - верхней границы диапазона (последовательности должны быть одной длины!)
    # Входная последовательность: символы с i по i + SEQ_LENGTH - 1
    input_seq = data_indices[i:i + SEQ_LENGTH] # 50 символов, начиная с позиции i
    # Целевая последовательность: символы со сдвигом на 1 (с i+1 по i + SEQ_LENGTH)
    target_seq = data_indices[i + 1:i + 1 + SEQ_LENGTH] # те же 50 символов, но сдвинутые на 1 позицию вперёд (цель — предсказать следующий символ)

    input_sequences.append(input_seq)
    target_sequences.append(target_seq)
print(f'Последовательности одной длины? - {len(input_sequences) == len(target_sequences)}')

Последовательности одной длины? - True


In [14]:
# Конвертация последовательностей в тензоры!
X_tensor_input = torch.tensor(input_sequences, dtype=torch.long) # long (целочисленные)
y_tensor_target = torch.tensor(target_sequences, dtype=torch.long)

In [15]:
print(X_tensor_input.shape)
print(y_tensor_target.shape)

torch.Size([515816, 50])
torch.Size([515816, 50])


In [16]:
from torch.utils.data import TensorDataset, DataLoader
# Создание DataLoader
# 1. Сначало создаем объект TensorDataset из тензоров последовательностей
dataset_tensor = TensorDataset(X_tensor_input, y_tensor_target)
print(dataset_tensor.tensors[0].shape)

torch.Size([515816, 50])


In [17]:
# 2. # Создаём DataLoader с перемешиванием
dataloader = DataLoader(
    dataset_tensor,
    batch_size=BATCH_SIZE,
    shuffle=True
)

In [18]:
# ИТОГ:
for x, y in dataloader:
  print(f"Размер батча: {x.shape}")
  print(f"Число символов в одной последовательности: {x[0].shape[0]}")
  break  # Выходим после первого батча

print(f'Общее количество батчей: {len(dataloader)}\n'
      f'Общее количество символов в dataloader: {len(dataloader.dataset)}\n')

Размер батча: torch.Size([64, 50])
Число символов в одной последовательности: 50
Общее количество батчей: 8060
Общее количество символов в dataloader: 515816



## Построение модели

In [19]:
import torch
import torch.nn as nn

class My_RNNModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers=1):
        super(My_RNNModel, self).__init__()
        self.hidden_size = hidden_size         # размер rnn скрытой ячейки (н-р 256)
        self.num_layers = num_layers           # число слоев

        # Слой эмбеддингов
        self.embedding = nn.Embedding(vocab_size, embedding_dim) # размер словаря и эмбединга (55 и 128)
        # Готовая RNN-ячейка PyTorch
        self.rnn = nn.RNN(embedding_dim, hidden_size, num_layers, batch_first=True)
        # Выходной полносвязный слой
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        batch_size = x.size(0)

        # Эмбеддинги
        embedded = self.embedding(x)  # (batch_size, seq_length, embedding_dim)

        # Пропускаем через RNN
        output, hidden = self.rnn(embedded, hidden)
        # Пропускаем через выходной слой
        output = self.fc(output)  # (batch_size, seq_length, vocab_size)
        return output, hidden

In [20]:
dictionary_size # символов в словаре

55

In [21]:
# Создаю модель
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = My_RNNModel(vocab_size = dictionary_size, embedding_dim = 128, hidden_size=256, num_layers=2).to(device)

In [22]:
model

My_RNNModel(
  (embedding): Embedding(55, 128)
  (rnn): RNN(128, 256, num_layers=2, batch_first=True)
  (fc): Linear(in_features=256, out_features=55, bias=True)
)

## Обучение модели

In [29]:
# Параметры модели
LEARNING_RATE = 0.001
EPOCHS = 10

# Функция потерь и оптимизатор
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Обучение с детализированным выводом
model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    total_correct = 0   # Корректно предсказанных символов
    total_elements = 0  # Общее количество предсказанных символов

    for batch_idx, (data, target) in enumerate(dataloader):
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()

        output, _ = model(data)
        # Вычесление функции потерь
        loss = criterion(output.view(-1, dictionary_size), target.view(-1))

        # Расчёт accuracy с подсчётом всех элементов
        predictions = torch.argmax(output, dim=-1)  # (batch_size, seq_length)
        correct_predictions = (predictions == target).sum().item()  # Количество правильных предсказаний в батче
        total_elements_in_batch = target.numel()  # Общее количество элементов в батче

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_correct += correct_predictions
        total_elements += total_elements_in_batch

    # Итоговая accuracy за epoch (по всем символам)
    epoch_accuracy = total_correct / total_elements if total_elements > 0 else 0
    avg_loss = total_loss / len(dataloader)


    print(f'Epoch {epoch:3d} | Loss: {avg_loss:10.8f} | Accuracy: {epoch_accuracy:10.8f} '
         f'(Корректно предсказанных символов: {total_correct} / Общее количество предсказанных символов: {total_elements})')


Epoch   0 | Loss: 1.04220431 | Accuracy: 0.66693813 (Корректно предсказанных символов: 17200868/Общее количество предсказанных символов: 25790800)
Epoch   1 | Loss: 1.04165038 | Accuracy: 0.66702797 (Корректно предсказанных символов: 17203185/Общее количество предсказанных символов: 25790800)
Epoch   2 | Loss: 1.04156947 | Accuracy: 0.66699118 (Корректно предсказанных символов: 17202236/Общее количество предсказанных символов: 25790800)
Epoch   3 | Loss: 1.04150569 | Accuracy: 0.66708194 (Корректно предсказанных символов: 17204577/Общее количество предсказанных символов: 25790800)
Epoch   4 | Loss: 1.04100048 | Accuracy: 0.66730803 (Корректно предсказанных символов: 17210408/Общее количество предсказанных символов: 25790800)
Epoch   5 | Loss: 1.04121656 | Accuracy: 0.66707179 (Корректно предсказанных символов: 17204315/Общее количество предсказанных символов: 25790800)
Epoch   6 | Loss: 1.04081997 | Accuracy: 0.66732366 (Корректно предсказанных символов: 17210811/Общее количество предс

## Генерация текста. Стартовый текст предварительно должен быть нормализован!

In [72]:
def generate_text(model, start_text, length=30, temperature=0.5): # length - длина текста, temperature - креативность (0.5, 1.0, 2.0)-степень разнообразия
    model.eval()
    device = next(model.parameters()).device

    generated = list(start_text)
    current_input = torch.tensor([char_to_idx.get(ch, 0) for ch in start_text], # преобразование в индексы стартового текста в тензор и монтаж тензора в device
                               dtype=torch.long).unsqueeze(0).to(device)

    hidden = None

    with torch.no_grad():
        for _ in range(length):
            output, hidden = model(current_input, hidden)
            # нужен последний символ последовательности
            logits = output[0, -1, :] / temperature
            probs = torch.softmax(logits, dim=-1)
            # Сэмплируется следующий символ
            next_idx = torch.multinomial(probs, 1).item()
            next_char = idx_to_char[next_idx]

            generated.append(next_char)

            # Обновляем вход для следующего шага
            current_input = torch.tensor([[next_idx]], dtype=torch.long).to(device)

    return ''.join(generated)

In [37]:
df.columns

Index(['Unnamed: 0', 'id', 'episode_id', 'number', 'raw_text',
       'timestamp_in_ms', 'speaking_line', 'character_id', 'location_id',
       'raw_character_text', 'raw_location_text', 'spoken_words',
       'normalized_text', 'word_count'],
      dtype='object')

In [66]:
# Самая длинная фраза
longest_idx = df['normalized_text'].str.len().idxmax()
longest_phrase = df['normalized_text'].iloc[longest_idx]
print(longest_phrase)
print(f'Самая динная фраза c индексом - {longest_idx} в количестве символов: {len(longest_phrase)}')

elegy for geezer rock postcard image thing to see to think of springfield is to think of thee what thoughts be-pass ahind thy mien why sky art blue why trees art green and what pray tell did thine eyes see perchance old friend they gazed at me brought low by natures oafish hand thou crush-ed our reviewing stand and twixt thy stones glimpsed i the truth all things must pass -- thy face my youth
Самая динная фраза c индексом - 5381 в количестве символов: 396


In [48]:
# Примеры стартовых фраз/ Взял рандомно из датасета
import random
if 'normalized_text' in df.columns: # Проверяем, что столбец существует(т.к присутсвуют пустоты)
    # Беру 5 случайных фраз
    random_phrases_start = random.sample(df['normalized_text'].dropna().tolist(), 5)

In [77]:
random_phrases_start

['all right its all hooked up',
 'uh dad they turned you into a vampire',
 'you mean like when you hid inside the conga drum to scare ricky',
 'nelson youre not really angry at me youre full of rage cause your father abandoned you',
 'sideshow bob']

In [78]:
# Пример сгенерированной 3 фразы из списка
generate_text(model=model, start_text=random_phrases_start[2])

'you mean like when you hid inside the conga drum to scare ricky oh i dont know what have a ba'